In [ ]:
import argparse
from typing import NamedTuple

import gym
import numpy as np
import torch
import torch.optim as optim
# !pip install torchopt

import torchopt
import sys

from google.colab import drive

drive.mount('/content/drive')
import sys
sys.path.append('/content/drive')

import policy
import tabular_mdp


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch.nn as nn
from torch.distributions import Categorical


class CategoricalMLPPolicy(nn.Module):
    """Policy network based on a multi-layer perceptron (MLP), with a
    `Categorical` distribution output. This policy network can be used on tasks
    with discrete action spaces (eg. `TabularMDPEnv`).
    """

    def __init__(self, input_size, output_size):
        super().__init__()
        self.torso = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
        )
        self.policy_head = nn.Linear(32, output_size)
        self.value_head = nn.Linear(32, 1)

    def forward(self, inputs, params=None):
        embedding = self.torso(inputs)
        logits = self.policy_head(embedding)
        values = self.value_head(embedding)
        return Categorical(logits=logits), values

In [ ]:
class Traj(NamedTuple):
    obs: np.ndarray
    acs: np.ndarray
    next_obs: np.ndarray
    rews: np.ndarray
    gammas: np.ndarray


def sample_traj(env, task, policy):
    env.reset_task(task)
    obs_buf = np.zeros(shape=(TRAJ_LEN, TRAJ_NUM, STATE_DIM), dtype=np.float32)
    next_obs_buf = np.zeros(shape=(TRAJ_LEN, TRAJ_NUM, STATE_DIM), dtype=np.float32)
    acs_buf = np.zeros(shape=(TRAJ_LEN, TRAJ_NUM), dtype=np.int8)
    rews_buf = np.zeros(shape=(TRAJ_LEN, TRAJ_NUM), dtype=np.float32)
    gammas_buf = np.zeros(shape=(TRAJ_LEN, TRAJ_NUM), dtype=np.float32)
    with torch.no_grad():
        for batch in range(TRAJ_NUM):
            ob = env.reset()
            for step in range(TRAJ_LEN):
                ob_tensor = torch.from_numpy(ob)
                pi, _ = policy(ob_tensor)
                ac_tensor = pi.sample()
                ac = ac_tensor.cpu().numpy()
                next_ob, rew, done, info = env.step(ac)

                obs_buf[step][batch] = ob
                next_obs_buf[step][batch] = next_ob
                acs_buf[step][batch] = ac
                rews_buf[step][batch] = rew
                gammas_buf[step][batch] = (1 - done) * GAMMA
                ob = next_ob
    return Traj(
        obs=obs_buf,
        acs=acs_buf,
        next_obs=next_obs_buf,
        rews=rews_buf,
        gammas=gammas_buf,
    )

In [ ]:
def a2c_loss(traj, policy, value_coef):
    lambdas = np.ones_like(traj.gammas) * LAMBDA
    _, next_values = policy(torch.from_numpy(traj.next_obs))
    next_values = torch.squeeze(next_values, -1).detach().numpy()
    # Work backwards to compute `G_{T-1}`, ..., `G_0`.
    returns = []
    g = next_values[-1, :]
    for i in reversed(range(next_values.shape[0])):
        g = traj.rews[i, :] + traj.gammas[i, :] * (
            (1 - lambdas[i, :]) * next_values[i, :] + lambdas[i, :] * g
        )
        returns.insert(0, g)
    lambda_returns = torch.from_numpy(np.array(returns))
    pi, values = policy(torch.from_numpy(traj.obs))
    log_probs = pi.log_prob(torch.from_numpy(traj.acs))
    advs = lambda_returns - torch.squeeze(values, -1)
    action_loss = -(advs.detach() * log_probs).mean()
    value_loss = advs.pow(2).mean()

    loss = action_loss + value_coef * value_loss
    return loss


def evaluate(env, seed, task_num, policy):
    pre_reward_ls = []
    post_reward_ls = []
    inner_opt = torchopt.MetaSGD(policy, lr=0.1)
    env = gym.make(
        'tabular_mdp:TabularMDP-v0',
        num_states=STATE_DIM,
        num_actions=ACTION_DIM,
        max_episode_steps=TRAJ_LEN,
        seed=args.seed,
    )
    tasks = env.sample_tasks(num_tasks=task_num)
    policy_state_dict = torchopt.extract_state_dict(policy)
    optim_state_dict = torchopt.extract_state_dict(inner_opt)
    for idx in range(task_num):
        for _ in range(inner_iters):
            pre_trajs = sample_traj(env, tasks[idx], policy)
            inner_loss = a2c_loss(pre_trajs, policy, value_coef=0.5)
            inner_opt.step(inner_loss)
        post_trajs = sample_traj(env, tasks[idx], policy)

        # Logging
        pre_reward_ls.append(np.sum(pre_trajs.rews, axis=0).mean())
        post_reward_ls.append(np.sum(post_trajs.rews, axis=0).mean())

        torchopt.recover_state_dict(policy, policy_state_dict)
        torchopt.recover_state_dict(inner_opt, optim_state_dict)
    return pre_reward_ls, post_reward_ls


In [ ]:
def main(args):

    # init training
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    # Env
    env = gym.make(
        'tabular_mdp:TabularMDP-v0',
        num_states=STATE_DIM,
        num_actions=ACTION_DIM,
        max_episode_steps=TRAJ_LEN,
        seed=args.seed,
    )
    # Policy
    policy = CategoricalMLPPolicy(input_size=STATE_DIM, output_size=ACTION_DIM)
    inner_opt = torchopt.MetaSGD(policy, lr=0.1)
    outer_opt = optim.Adam(policy.parameters(), lr=1e-3)
    train_pre_reward = []
    train_post_reward = []
    test_pre_reward = []
    test_post_reward = []

    for i in range(outer_iters):
        tasks = env.sample_tasks(num_tasks=TASK_NUM)
        train_pre_reward_ls = []
        train_post_reward_ls = []

        outer_opt.zero_grad()

        policy_state_dict = torchopt.extract_state_dict(policy)
        optim_state_dict = torchopt.extract_state_dict(inner_opt)
        for idx in range(TASK_NUM):
            for _ in range(inner_iters):
                pre_trajs = sample_traj(env, tasks[idx], policy)
                inner_loss = a2c_loss(pre_trajs, policy, value_coef=0.5)
                inner_opt.step(inner_loss)
            post_trajs = sample_traj(env, tasks[idx], policy)
            outer_loss = a2c_loss(post_trajs, policy, value_coef=0.5)
            outer_loss.backward()
            torchopt.recover_state_dict(policy, policy_state_dict)
            torchopt.recover_state_dict(inner_opt, optim_state_dict)
            # Logging
            train_pre_reward_ls.append(np.sum(pre_trajs.rews, axis=0).mean())
            train_post_reward_ls.append(np.sum(post_trajs.rews, axis=0).mean())
        outer_opt.step()

        test_pre_reward_ls, test_post_reward_ls = evaluate(env, args.seed, TASK_NUM, policy)

        train_pre_reward.append(sum(train_pre_reward_ls) / TASK_NUM)
        train_post_reward.append(sum(train_post_reward_ls) / TASK_NUM)
        test_pre_reward.append(sum(test_pre_reward_ls) / TASK_NUM)
        test_post_reward.append(sum(test_post_reward_ls) / TASK_NUM)

        print('Train_iters', i)
        print('train_pre_reward', sum(train_pre_reward_ls) / TASK_NUM)
        print('train_post_reward', sum(train_post_reward_ls) / TASK_NUM)
        print('test_pre_reward', sum(test_pre_reward_ls) / TASK_NUM)
        print('test_post_reward', sum(test_post_reward_ls) / TASK_NUM)


In [ ]:
TASK_NUM = 40
TRAJ_NUM = 20
TRAJ_LEN = 10

STATE_DIM = 10
ACTION_DIM = 5

GAMMA = 0.99
LAMBDA = 0.95

outer_iters = 500
inner_iters = 1

In [ ]:
from gym.envs.registration import register


register(
    'TabularMDP-v0',
    entry_point='tabular_mdp:TabularMDPEnv',
    kwargs={'num_states': 10, 'num_actions': 5, 'max_episode_steps': 10, 'seed': 1},
)

/usr/local/lib/python3.10/dist-packages/gym/envs/registration.py:542: UserWarning: WARN: Overriding environment TabularMDP-v0
  logger.warn(f"Overriding environment {spec.id}")


In [ ]:
if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Example script')
    parser.add_argument('--seed', type=int, default=1, help='Random seed')
    args, unknown = parser.parse_known_args()


    main(args)

/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:190: UserWarning: WARN: Future gym versions will require 

Train_iters 0
train_pre_reward 10.14358080625534
train_post_reward 9.915057861804963
test_pre_reward 9.756197929382324
test_post_reward 9.899411392211913
Train_iters 1
train_pre_reward 9.82133400440216
train_post_reward 10.030324268341065
test_pre_reward 9.86304830312729
test_post_reward 9.92957216501236
Train_iters 2
train_pre_reward 9.893833756446838
train_post_reward 10.0176593542099
test_pre_reward 9.72222067117691
test_post_reward 9.828013586997987
Train_iters 3
train_pre_reward 10.043111717700958
train_post_reward 9.744170486927032
test_pre_reward 10.089351797103882
test_post_reward 10.142765116691589
Train_iters 4
train_pre_reward 9.972537517547607
train_post_reward 9.89646382331848
test_pre_reward 9.72798320055008
test_post_reward 10.009267044067382
Train_iters 5
train_pre_reward 10.045961368083955
train_post_reward 10.226897740364075
test_pre_reward 9.930195271968842
test_post_reward 9.9702805519104
Train_iters 6
train_pre_reward 9.83591216802597
train_post_reward 10.229260909

KeyboardInterrupt: 